# 11-phase12-hybrid-graph-foundations

**neuron Phase 12** — paradigm 의 ultimate 단계. Phase 9 group + Phase 10/11 channel 의
계층적 hybrid (사용자 vision: 히든 레이어 = graph 의 ultimate 구현).

핵심 가설:
1. **function preservation** — hybrid_full_full (outer=full, inner=full) ≈ plain Linear?
2. **Phase 9/10/11 통합 표현** — hybrid 의 init 조합으로 이전 phase 결과 재현 가능?
   - outer=identity + inner=full ≈ Phase 9 group_identity (+0.14 열위)
   - outer=full + inner=around_one ≈ Phase 11 channel_around_one (≈ plain)
3. **계층적 routing 효과** — 두 adj 의 학습된 패턴 분리 시각화?
4. **paradigm 안정성** — magnitude rule 자동 적용으로 모든 init 조합 안전?

설계: 4 × 2 sweep = 8 run, max_steps=1500.

데이터: TinyShakespeare (char-LM)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#65](https://github.com/EinSofINTEREST/GraphLM/issues/65) / Phase 11 baseline PR [#64](https://github.com/EinSofINTEREST/GraphLM/pull/64)

## 1. 환경

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.graph_hybrid_demo import train_hybrid_graph_mlp
from graphlm.utils import repo_root

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)
print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

In [ ]:
import math

ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase12"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123]
ARCHS = [
    "plain",
    "hybrid_full_full",
    "hybrid_identity_full",
    "hybrid_full_around_one",
]
EMB_DIM = 64
HIDDEN_DIM = 256  # 256 / 16 = 16 groups
GROUP_SIZE = 16
N_GRAM = 4
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500

# Phase 9/10/11 baseline (직접 비교)
PHASE9_GROUP_FULL = 2.1391
PHASE9_GROUP_IDENTITY = 2.2797  # 0-init vanishing 사례
PHASE10_CHANNEL_FULL = 2.1339
PHASE11_CHANNEL_AROUND_ONE = 2.1456

print(f"SEEDS      = {SEEDS}")
print(f"ARCHS      = {ARCHS}")
print(
    f"HIDDEN_DIM = {HIDDEN_DIM}, GROUP_SIZE = {GROUP_SIZE} → n_groups = {HIDDEN_DIM // GROUP_SIZE}"
)
print(f"MAX_STEPS  = {MAX_STEPS}")
print(f"전체 run   = {len(SEEDS) * len(ARCHS)}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
V = tokenizer.vocab_size

V_PADDED = math.ceil(V / GROUP_SIZE) * GROUP_SIZE
print(f"vocab_size : {V}, padded to {V_PADDED} (GROUP_SIZE {GROUP_SIZE} 배수)")

## 4. sweep 학습

In [ ]:
runs = {}
for seed in SEEDS:
    for arch in ARCHS:
        key = (seed, arch)
        print(f"--- seed={seed}, arch={arch} ---")
        runs[key] = train_hybrid_graph_mlp(
            dataset=dataset,
            vocab_size=V_PADDED,
            seed=seed,
            arch=arch,
            emb_dim=EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            group_size=GROUP_SIZE,
            n_gram=N_GRAM,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            device=DEVICE,
        )
        print(f"  done: final_loss={runs[key]['final_loss']:.4f}")
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 + Phase 9/10/11 baseline 비교

In [ ]:
import statistics

print(f"{'arch':>28}  {'seed':>5}  {'final_loss':>11}")
print("-" * 55)
for arch in ARCHS:
    for seed in SEEDS:
        r = runs[(seed, arch)]
        print(f"{arch:>28}  {seed:>5}  {r['final_loss']:>11.4f}")

print()
print("=== arch 별 mean ===")
agg = {}
for arch in ARCHS:
    fls = [runs[(s, arch)]["final_loss"] for s in SEEDS]
    agg[arch] = dict(mean=statistics.mean(fls), range=max(fls) - min(fls))
    print(f"  {arch:>28}: mean={agg[arch]['mean']:.4f}, range={agg[arch]['range']:.4f}")

print()
print("=== Phase 9/10/11 baseline (직접 비교 — 통합 표현 검증) ===")
print(f"  Phase 9  group_full          : {PHASE9_GROUP_FULL:.4f}")
print(f"  Phase 9  group_identity      : {PHASE9_GROUP_IDENTITY:.4f}  (0-init vanishing 사례)")
print(f"  Phase 10 channel_full        : {PHASE10_CHANNEL_FULL:.4f}")
print(f"  Phase 11 channel_around_one  : {PHASE11_CHANNEL_AROUND_ONE:.4f}")
print()
print(f"  Phase 12 hybrid_full_full         : {agg['hybrid_full_full']['mean']:.4f}")
print(
    f"  Phase 12 hybrid_identity_full     : {agg['hybrid_identity_full']['mean']:.4f}  (expected ≈ Phase 9 group_identity)"
)
print(
    f"  Phase 12 hybrid_full_around_one   : {agg['hybrid_full_around_one']['mean']:.4f}  (expected ≈ Phase 11 channel_around_one)"
)

print()
print("=== 자동 verdict ===")
plain = agg["plain"]["mean"]
full_full = agg["hybrid_full_full"]["mean"]
identity_full = agg["hybrid_identity_full"]["mean"]
full_around = agg["hybrid_full_around_one"]["mean"]
range_max = max(agg[a]["range"] for a in ARCHS)
print(f"  range_max = {range_max:.4f}")
print(f"  hybrid_full_full vs plain         : {full_full - plain:+.4f}")
print(
    f"  hybrid_identity_full vs plain     : {identity_full - plain:+.4f} (expected ~+0.14 like Phase 9)"
)
print(f"  hybrid_full_around_one vs plain   : {full_around - plain:+.4f} (expected ≈ 0)")
if abs(full_full - plain) < range_max:
    print("  ✅ hybrid_full_full ≈ plain — function preservation 작동")
if identity_full - plain > 0.05:
    print("  ✅ hybrid_identity_full 열위 — Phase 9 group_identity 패턴 재현")
if abs(full_around - plain) < range_max:
    print("  ✅ hybrid_full_around_one ≈ plain — Phase 11 channel_around_one 패턴 재현")

## 6. 학습된 adj_outer + adj_inner 시각화 (hybrid_full_full)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

r = runs[(SEEDS[0], "hybrid_full_full")]
adj_outer = r["final_adj"]["outer"].numpy()
adj_inner = r["final_adj"]["inner"].numpy()
print(f"adj_outer shape: {adj_outer.shape}")
print(f"adj_inner shape: {adj_inner.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) adj_outer heatmap
vmax_o = max(abs(adj_outer).max(), 1e-6)
im0 = axes[0].imshow(adj_outer, cmap="RdBu_r", vmin=-vmax_o, vmax=vmax_o, aspect="auto")
axes[0].set_xlabel("input group")
axes[0].set_ylabel("output group")
axes[0].set_title(
    f"adj_outer (group-level)\nmean={adj_outer.mean():.3f}, std={adj_outer.std():.3f}"
)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# (b) adj_inner mean abs by (G_out, G_in) — block-aggregated heatmap
inner_block_mean = np.abs(adj_inner).mean(axis=(2, 3))
vmax_i = max(inner_block_mean.max(), 1e-6)
im1 = axes[1].imshow(inner_block_mean, cmap="viridis", vmin=0, vmax=vmax_i, aspect="auto")
axes[1].set_xlabel("input group")
axes[1].set_ylabel("output group")
axes[1].set_title(f"adj_inner |mean| per block\noverall mean={inner_block_mean.mean():.3f}")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle(f"Phase 12 — hybrid_full_full 학습된 adj (seed={SEEDS[0]}, fc1)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "hybrid_adj.png", dpi=120)
plt.show()

## 7. loss curve 4 arch 비교 (mean ± σ)

In [ ]:
window = 30
colors = {
    "plain": "#1f77b4",
    "hybrid_full_full": "#2ca02c",
    "hybrid_identity_full": "#d62728",
    "hybrid_full_around_one": "#ff7f0e",
}

fig, ax = plt.subplots(figsize=(13, 5))
for arch in ARCHS:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(seed, arch)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    ax.plot(steps, mean, color=colors[arch], lw=1.5, label=arch)
    ax.fill_between(steps, mean - std, mean + std, color=colors[arch], alpha=0.15)

ax.set_xlabel("step")
ax.set_ylabel(f"loss (smoothed window={window})")
ax.set_title("Phase 12 — 4 arch loss curve (mean ± σ across 2 seeds)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curves.png", dpi=120)
plt.show()

## 결과 요약 / Phase 13 권장

확인 포인트:
- §5 hybrid_full_full ≈ plain (function preservation)
- §5 hybrid_identity_full vs Phase 9 group_identity (+0.14 패턴 재현)
- §5 hybrid_full_around_one vs Phase 11 channel_around_one (≈ plain 패턴 재현)
- §6 학습된 adj_outer + adj_inner 의 분리 표현
- §7 4 arch 의 발산/수렴 패턴

**판정 시나리오**:
- **A. 4 arch 모두 expected 와 일치** ⭐ — hybrid 가 Phase 9/10/11 의 통합 표현 입증 → Phase 13 (Transformer 통합) 진입
- **B. 일부 불일치** — hybrid 의 dual adj 의 interaction effect 가 단순 합 이상의 dynamics
- **C. hybrid_full_full 도 plain 보다 명확 우위** — 계층적 routing 의 추가 표현력 발견 (surprise positive)

**참고**:
- Phase 11 결과: https://www.notion.so/36ce8b70b7aa81b4b684f9c559788a46
- 아키텍처 구성 계획: https://www.notion.so/36ce8b70b7aa818cbf1fe71687b449b8